<a href="https://colab.research.google.com/github/Dineshbp-10/mistral_med/blob/main/mistral_med.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers accelerate bitsandbytes sentencepiece

In [ ]:
!pip install huggingface_hub



In [ ]:
!pip install ollama

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Define a sample sentence
sample_sentence = "Hello, how are you doing today?"

# Tokenize the sentence
tokens = tokenizer.tokenize(sample_sentence)

# Print the tokens
print("Original Sentence:", sample_sentence)
print("Tokens:", tokens)

# You can also get input IDs, which are numerical representations of tokens
input_ids = tokenizer.encode(sample_sentence, add_special_tokens=True)
print("Input IDs (with special tokens):", input_ids)
print("Decoded Input IDs:", tokenizer.decode(input_ids))

In [ ]:
from huggingface_hub import hf_hub_download

# Download the 4-bit optimized medical model (~4.1GB)
model_path = hf_hub_download(
    repo_id="MaziyarPanahi/BioMistral-7B-GGUF",
    filename="BioMistral-7B.Q4_K_M.gguf"
)
print(f"Model downloaded successfully to: {model_path}")


In [ ]:
!pip install ctransformers

In [ ]:
m_path=r'/root/.cache/huggingface/hub/models--MaziyarPanahi--BioMistral-7B-GGUF/snapshots/ae4f07544f1015dc8f5bf382b7582852638cbecf/BioMistral-7B.Q4_K_M.gguf'

In [ ]:
from ctransformers import AutoModelForCausalLM

path=m_path
print("Loading healthcare model into RAM...")

# Load the model (Configured for 16GB RAM)
llm = AutoModelForCausalLM.from_pretrained(path, model_type="mistral",hf=False)

# Ask a medical question
prompt = "Explain the symptoms of Leukemia."
response = llm(prompt, max_new_tokens=259)

print("\n--- Medical Response ---\n")
print(response)


In [ ]:
from ctransformers import AutoModelForCausalLM
from transformers import AutoTokenizer


# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("BioMistral/BioMistral-7B")

path=m_path
print("Loading BioMistral...")

# --- 2. LOADING THE MODEL ---
# AutoModelForCausalLM loads the model architecture into RAM.
# model_type="mistral" tells the library which specific logic to use.
# hf=False tells it you are using a local CTransformers model, not a standard HuggingFace PyTorch model.

llm = AutoModelForCausalLM.from_pretrained(path, model_type="mistral", hf=False)

print("\n--- Chat Started (Type 'exit' to stop) ---")

# --- 3. THE CHAT LOOP ---
while True:
    # Get user input from the console
    user_input = input("\nYou: ")

    # Check if the user wants to quit
    if user_input.lower() in ["exit", "quit"]:
        break

    # Format the input so the AI knows it's a conversation
    # prompt = f"User: {user_input}\nAssistant:"

    prompt = f"<s>[INST] {user_input} [/INST]"
    user_token_ids = tokenizer.encode(prompt)
    user_count = len(user_token_ids)

    print(f" (User Tokens: {user_count})")
    print("\nAssistant: ", end="", flush=True)

    full_assistant_response = ""
    # --- 4. GENERATING THE RESPONSE ---
    # max_new_tokens: Limits how long the AI's answer can be.
    # stream=True: This is the 'magic' that makes text appear word-by-word.
    # It returns a 'generator' that we loop through to print text as it's created.

    for text in llm(prompt, max_new_tokens=512, stream=True,temperature=0.2, top_p=0.95):
        print(text, end="", flush=True)
        full_assistant_response += text

# 3. Count Assistant Tokens
    # We tokenize the full string at the end for accuracy
    assistant_token_ids = tokenizer.encode(full_assistant_response, add_special_tokens=False)
    assistant_count = len(assistant_token_ids)

    print(f"\n\n[Token Usage -> User: {user_count} | Assistant: {assistant_count} | Total: {user_count + assistant_count}]")

